# Herb Awareness Survey

Exploratory data analysis and feature engineering using the herb awareness survey responses.

## 1. Load the data

In [1]:
import pandas as pd
from IPython.display import display

google_sheet_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTicNSlQXwx8x-lA8WTUD_xs9gIWTwMMhuvYH9igsTWO7v89bglov0B6OaRV0XKzFvgYAARNVt0Pap4/pub?output=csv'
df = pd.read_csv(google_sheet_url)
print("Responses:", df.shape[0])
print("Columns:", df.shape[1])

Responses: 28
Columns: 10


## 2. Explore and prepare the data

In [2]:
# Remove accidental spaces from the column headings.
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
display(df.dtypes.to_frame("Data type"))
display(df.isna().sum().to_frame("Missing values"))

# Assign a row identifier for counting responses, not verified individuals.
df.insert(0, "Response ID", range(1, len(df) + 1))

# Remove names and use shorter question headings.
df = df.drop(columns=["Q1. Full Name"], errors="ignore")
df = df.rename(columns={
    "Q2. Gender": "Gender",
    "Q3. Age": "Age",
    "Q4. Have you ever heard about medicinal herbs?": "Herbal Awareness",
    "Q5. How would you rate your knowledge of herbs?": "Knowledge Level",
    "Q6. Have you ever used herbs for health, wellness, cooking or skincare purposes?": "Usage Frequency",
    "Q7. Where do you mainly get information about herbs? (Select one)": "Information Source",
    "Q8. What do you mainly associate herbs with? (Select one)": "Primary Association",
    "Q9. Would you consider buying herb-based products in the future?": "Purchase Intention"
})

for column in df.select_dtypes(include="object"):
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)

df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Knowledge Level"] = pd.to_numeric(df["Knowledge Level"], errors="coerce")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")

print("Exact duplicate rows:", df.duplicated().sum())
# Retain responses: matching answers alone do not establish repeat participation.
display(df.drop(columns="Timestamp").head())
display(df[["Age", "Knowledge Level"]].describe())

print("Ages outside 0–120:", (~df["Age"].between(0, 120) & df["Age"].notna()).sum())
print("Knowledge scores outside 1–5:", (~df["Knowledge Level"].isin([1, 2, 3, 4, 5]) & df["Knowledge Level"].notna()).sum())
print("Awareness answers:", df["Herbal Awareness"].dropna().unique())
print("Usage answers:", df["Usage Frequency"].dropna().unique())
print("Purchase answers:", df["Purchase Intention"].dropna().unique())

,Data type
Timestamp,object
Q1. Full Name,object
Q2. Gender,object
Q3. Age,int64
Q4. Have you ever heard about medicinal herbs?,object
Q5. How would you rate your knowledge of herbs?,int64
"Q6. Have you ever used herbs for health, wellness, cooking or skincare purposes?",object
Q7. Where do you mainly get information about herbs? (Select one),object
Q8. What do you mainly associate herbs with? (Select one),object
Q9. Would you consider buying herb-based products in the future?,object


,Missing values
Timestamp,0
Q1. Full Name,2
Q2. Gender,0
Q3. Age,0
Q4. Have you ever heard about medicinal herbs?,0
Q5. How would you rate your knowledge of herbs?,0
"Q6. Have you ever used herbs for health, wellness, cooking or skincare purposes?",0
Q7. Where do you mainly get information about herbs? (Select one),0
Q8. What do you mainly associate herbs with? (Select one),0
Q9. Would you consider buying herb-based products in the future?,0


Exact duplicate rows: 0


,Response ID,Gender,Age,Herbal Awareness,Knowledge Level,Usage Frequency,Information Source,Primary Association,Purchase Intention
0,1,Male,27,Yes,4,"Yes, occasionally",Family/Friends,Improving health and wellness,Definitely
1,2,Male,27,Yes,5,"Yes, occasionally",Internet/Websites,Cooking and food flavour enhancement,Probably
2,3,Female,27,Yes,3,"Yes, frequently","Family/Friends, Social Media, Internet/Website...",Cooking and food flavour enhancement,Probably
3,4,Female,23,Yes,3,"Yes, occasionally",Social Media,Cooking and food flavour enhancement,Probably
4,5,Female,25,Yes,3,"Yes, occasionally",Internet/Websites,Cooking and food flavour enhancement,Definitely


,Age,Knowledge Level
count,28.000000,28.000000
mean,31.214286,2.892857
std,8.757461,1.065947
min,19.000000,1.000000
25%,26.000000,2.000000
50%,27.000000,3.000000
75%,35.000000,3.000000
max,53.000000,5.000000


Ages outside 0–120: 0
Knowledge scores outside 1–5: 0
Awareness answers: <StringArray>
['Yes', 'No']
Length: 2, dtype: string
Usage answers: <StringArray>
['Yes, occasionally', 'Yes, frequently', 'No']
Length: 3, dtype: string
Purchase answers: <StringArray>
['Definitely', 'Probably', 'Probably not', 'Not sure', 'Definitely not']
Length: 5, dtype: string


## 3. Awareness, knowledge and usage

In [3]:
print("Awareness of medicinal herbs")
display(df["Herbal Awareness"].value_counts(dropna=False).to_frame("Respondents"))
display((df["Herbal Awareness"].value_counts(normalize=True) * 100).round(1).to_frame("Percent"))

print("Self-rated knowledge")
display(df["Knowledge Level"].value_counts().sort_index().to_frame("Respondents"))

print("Herb use")
display(df["Usage Frequency"].value_counts(dropna=False).to_frame("Respondents"))
display((df["Usage Frequency"].value_counts(normalize=True) * 100).round(1).to_frame("Percent"))

Awareness of medicinal herbs


,Respondents
Herbal Awareness,
Yes,23
No,5


,Percent
Herbal Awareness,
Yes,82.1
No,17.9


Self-rated knowledge


,Respondents
Knowledge Level,
1,3
2,5
3,15
4,2
5,3


Herb use


,Respondents
Usage Frequency,
"Yes, occasionally",13
No,8
"Yes, frequently",7


,Percent
Usage Frequency,
"Yes, occasionally",46.4
No,28.6
"Yes, frequently",25.0


**Insight:** Awareness is common (82.1%), but frequent herb use is less common (25.0%). The median self-rated knowledge score is 3 out of 5.

## 4. Information sources and main associations

In [4]:
# Actual answers include several sources, despite the single-choice wording.
sources = df[["Response ID", "Information Source"]].copy()
sources["Information Source"] = sources["Information Source"].str.split(",")
sources = sources.explode("Information Source")
sources["Information Source"] = sources["Information Source"].str.strip()
sources = sources.dropna()
sources = sources[sources["Information Source"] != ""].drop_duplicates()

source_summary = sources["Information Source"].value_counts().to_frame("Respondents mentioning")
source_summary["Percent"] = (source_summary["Respondents mentioning"] / df["Information Source"].count() * 100).round(1)
display(source_summary)

interest_counts = df["Primary Association"].value_counts()
interest_percent = (df["Primary Association"].value_counts(normalize=True) * 100).round(1)
display(pd.DataFrame({"Respondents": interest_counts, "Percent": interest_percent}))

,Respondents mentioning,Percent
Information Source,,
Social Media,15,53.6
Family/Friends,11,39.3
Internet/Websites,7,25.0
School/ University,5,17.9
I've never came across this information,1,3.6


,Respondents,Percent
Primary Association,,
Improving health and wellness,14,50.0
Cooking and food flavour enhancement,8,28.6
I do not associate herbs with anything specific,2,7.1
Cultural/traditional practices,2,7.1
Natural alternative products,1,3.6
Beauty and skincare,1,3.6


**Insight:** Social media and family/friends are the most mentioned information sources. Health and wellness (50.0%) and cooking (28.6%) are the leading herb associations.

## 5. Future purchase intention

In [5]:
purchase_counts = df["Purchase Intention"].value_counts().reindex(
    ["Definitely", "Probably", "Not sure", "Probably not", "Definitely not"], fill_value=0)
purchase_percent = (purchase_counts / df["Purchase Intention"].count() * 100).round(1)
display(pd.DataFrame({"Respondents": purchase_counts, "Percent": purchase_percent}))

,Respondents,Percent
Purchase Intention,,
Definitely,15,53.6
Probably,10,35.7
Not sure,1,3.6
Probably not,1,3.6
Definitely not,1,3.6


**Insight:** 25 of 28 respondents (89.3%) would definitely or probably consider buying herb-based products.

## 6. Feature engineering

### Age groups

In [6]:
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 17, 24, 34, 54, float("inf")],
    labels=["Under 18", "Young adults (18–24)", "Adults (25–34)",
            "Middle-aged adults (35–54)", "Older adults (55+)"],
    include_lowest=True
)
display(df["Age Group"].value_counts().sort_index().to_frame("Respondents"))

,Respondents
Age Group,
Under 18,0
Young adults (18–24),2
Adults (25–34),18
Middle-aged adults (35–54),8
Older adults (55+),0


### Awareness and usage

In [7]:
df["Aware"] = df["Herbal Awareness"].map({"Yes": "Aware", "No": "Not Aware"})
df["Usage_Score"] = df["Usage Frequency"].map({
    "No": 0, "Yes, occasionally": 1, "Yes, frequently": 2
})
# Usage_Score is an ordered code, not a measure of how many times herbs are used.
display(df[["Aware", "Knowledge Level", "Usage_Score"]].head())

,Aware,Knowledge Level,Usage_Score
0,Aware,4,1
1,Aware,5,1
2,Aware,3,2
3,Aware,3,1
4,Aware,3,1


### Potential customers

In [8]:
df["Potential_Customer"] = df["Purchase Intention"].map({
    "Definitely": "Potential Buyer",
    "Probably": "Potential Buyer",
    "Not sure": "Undecided",
    "Probably not": "Unlikely Buyer",
    "Definitely not": "Unlikely Buyer"
})
display(df["Potential_Customer"].value_counts(dropna=False).to_frame("Respondents"))

,Respondents
Potential_Customer,
Potential Buyer,25
Unlikely Buyer,2
Undecided,1


### Customer persona

In [9]:
def persona(row):
    if pd.isna(row["Aware"]) or pd.isna(row["Usage_Score"]):
        return "Incomplete / Review"
    if row["Aware"] == "Aware":
        if row["Usage_Score"] == 2:
            return "Aware / Frequent User"
        elif row["Usage_Score"] == 1:
            return "Aware / Occasional User"
        else:
            return "Aware / Non-user"
    elif row["Usage_Score"] > 0:
        return "Herb User / No Medicinal-herb Awareness"
    else:
        return "No Medicinal-herb Awareness / Non-user"

df["Customer_Persona"] = df.apply(persona, axis=1)
display(df["Customer_Persona"].value_counts().to_frame("Respondents"))

,Respondents
Customer_Persona,
Aware / Occasional User,11
Aware / Frequent User,6
Aware / Non-user,6
Herb User / No Medicinal-herb Awareness,3
No Medicinal-herb Awareness / Non-user,2


### Marketing priority

In [10]:
df["Marketing_Priority"] = df["Purchase Intention"].map({
    "Definitely": "Strong stated interest",
    "Probably": "Possible interest",
    "Not sure": "Explore uncertainty",
    "Probably not": "Low stated interest",
    "Definitely not": "Low stated interest"
})
display(df["Marketing_Priority"].value_counts(dropna=False).to_frame("Respondents"))

,Respondents
Marketing_Priority,
Strong stated interest,15
Possible interest,10
Low stated interest,2
Explore uncertainty,1


### Summary indicators
1 = yes/positive; 0 = other valid answers.

In [11]:
df["Ever Used Herbs"] = df["Usage Frequency"].map({
    "No": "No", "Yes, occasionally": "Yes", "Yes, frequently": "Yes"
})
df["Aware Flag"] = df["Herbal Awareness"].map({"Yes": 1, "No": 0})
df["Used Flag"] = df["Ever Used Herbs"].map({"Yes": 1, "No": 0})
df["Likely Purchase Flag"] = df["Potential_Customer"].map({
    "Potential Buyer": 1, "Undecided": 0, "Unlikely Buyer": 0
})

summary = pd.DataFrame({
    "Positive responses": df[["Aware Flag", "Used Flag", "Likely Purchase Flag"]].sum(),
    "Valid answers": df[["Aware Flag", "Used Flag", "Likely Purchase Flag"]].count(),
    "Percent": (df[["Aware Flag", "Used Flag", "Likely Purchase Flag"]].mean() * 100).round(1)
})
summary.index = ["Heard of medicinal herbs", "Ever used herbs", "Positive purchase intention"]
print("Total responses:", len(df))
display(summary)

Total responses: 28


,Positive responses,Valid answers,Percent
Heard of medicinal herbs,23,28,82.1
Ever used herbs,20,28,71.4
Positive purchase intention,25,28,89.3


## 7. Purchase interest by usage and association

In [12]:
# Compare interest among non-users, occasional users and frequent users.
usage_interest = pd.crosstab(df["Usage Frequency"], df["Potential_Customer"])
usage_interest = usage_interest.reindex(columns=["Potential Buyer", "Undecided", "Unlikely Buyer"], fill_value=0)
usage_interest["Valid intention answers"] = usage_interest.sum(axis=1)
usage_interest["Positive intention (%)"] = (
    usage_interest["Potential Buyer"] / usage_interest["Valid intention answers"] * 100).round(1)
display(usage_interest)

# See whether the two leading associations also coincide with purchase interest.
association_interest = pd.crosstab(df["Primary Association"], df["Potential_Customer"])
association_interest = association_interest.reindex(columns=["Potential Buyer", "Undecided", "Unlikely Buyer"], fill_value=0)
association_interest["Valid intention answers"] = association_interest.sum(axis=1)
association_interest["Positive intention (%)"] = (
    association_interest["Potential Buyer"] / association_interest["Valid intention answers"] * 100).round(1)
display(association_interest)

Potential_Customer,Potential Buyer,Undecided,Unlikely Buyer,Valid intention answers,Positive intention (%)
Usage Frequency,,,,,
No,6,0,2,8,75.0
"Yes, frequently",7,0,0,7,100.0
"Yes, occasionally",12,1,0,13,92.3


Potential_Customer,Potential Buyer,Undecided,Unlikely Buyer,Valid intention answers,Positive intention (%)
Primary Association,,,,,
Beauty and skincare,1,0,0,1,100.0
Cooking and food flavour enhancement,8,0,0,8,100.0
Cultural/traditional practices,1,0,1,2,50.0
I do not associate herbs with anything specific,0,1,1,2,0.0
Improving health and wellness,14,0,0,14,100.0
Natural alternative products,1,0,0,1,100.0


**Insight:** Positive purchase intention includes 7 of 7 frequent users, 12 of 13 occasional users and 6 of 8 non-users.

## 8. Export to Google Sheets
Run this cell in Google Colab and sign in. It creates a new spreadsheet with the analysed data, information sources and summary.

In [14]:
# Run this final cell in Google Colab.
!pip -q install gspread gspread-dataframe

from google.colab import auth
from google.auth import default
import gspread
from gspread_dataframe import set_with_dataframe

auth.authenticate_user()
credentials, _ = default()
gc = gspread.authorize(credentials)

# Export only after all feature engineering is complete.
df_export = df.drop(columns=["Timestamp"], errors="ignore").copy()

spreadsheet = gc.create("Herb Awareness Survey Analysis")
worksheet = spreadsheet.sheet1
worksheet.update_title("Analysed_Herb_Data")
set_with_dataframe(worksheet, df_export, include_index=False, resize=True)

# Keep source mentions separate so the main table stays one row per response.
source_export = sources.merge(
    df[["Response ID", "Gender", "Age Group", "Usage Frequency", "Potential_Customer"]],
    on="Response ID", how="left"
)
source_worksheet = spreadsheet.add_worksheet(title="Information_Sources", rows=100, cols=10)
set_with_dataframe(source_worksheet, source_export, include_index=False, resize=True)

summary_worksheet = spreadsheet.add_worksheet(title="Summary", rows=20, cols=5)
set_with_dataframe(summary_worksheet, summary.reset_index(names="Measure"), include_index=False, resize=True)

print("Analysed Herb data saved to Google Sheets:")
print(spreadsheet.url)

Analysed Herb data saved to Google Sheets:
https://docs.google.com/spreadsheets/d/1l2_liThuhVzktPU0pm51F7zF-xmkJrYArrEzSk9p-Pk
